In [1]:
# Requirements
##############
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Global variables
##################
RAW_CSV_FILE = "stamps_data_raw.csv"
CLEAN_CSV_FILE = "stamps_data_clean.csv"

In [3]:
# Load raw csv file
###################
import pandas as pd

df = pd.read_csv(RAW_CSV_FILE, sep="|")


In [4]:
# Check integrity of isue name. I missing values, report then and drop the rows
#
# The reason rows are dropped is that the issue name is a critical piece of 
# information for our analysis. If it's missing, it would be difficult to 
# categorize and analyze the data effectively. By dropping rows with missing 
# issue names, we ensure that our dataset remains clean and reliable for any 
# further analysis or modeling we may want to perform.
###############################################################################
missing_issue_names = df["issue_name"].isnull()
if missing_issue_names.any():
    print("🚨 Missing issue names:")
    print(df[missing_issue_names])
    df = df[~missing_issue_names]
else:
    print("✅ No missing issue names found.")


✅ No missing issue names found.


In [5]:
# Check integrity of issue datate. If missing values or not a valid date, report then, export the rows 
# to a csv file and drop them from the dataframe.
#
# The reason rows with missing or invalid issue dates are dropped is that the issue date is a crucial
# piece of information for our analysis. It allows us to understand the timeline of events and trends 
# in the data. If the issue date is missing or invalid, it would be difficult to analyze the data 
# accurately and could lead to incorrect conclusions. By dropping rows with missing or invalid issue 
# dates, we ensure that our dataset remains clean and reliable for any further analysis or modeling we 
# may want to perform. 
#
# Rows with missing or invalid issue dates are exported to a separate CSV file for further 
# investigation. This allows us to review the problematic entries and potentially correct them in the 
# future, while keeping our main dataset clean for analysis.
#######################################################################################################
missing_issue_dates = df["issue_date"].isnull()
invalid_issue_dates = pd.to_datetime(df["issue_date"], errors="coerce").isnull()
if missing_issue_dates.any() or invalid_issue_dates.any():
    print("🚨 Missing or invalid issue dates. Saving to missing_or_invalid_issue_dates.csv")
    df[missing_issue_dates | invalid_issue_dates].to_csv("missing_or_invalid_issue_dates.csv", sep='|', header=True, index=False)
    df = df[~(missing_issue_dates | invalid_issue_dates)]    
else:
    print("✅ No missing or invalid issue dates found.")

🚨 Missing or invalid issue dates. Saving to missing_or_invalid_issue_dates.csv


In [6]:
# Export all unique artists to a csv file. This will be used for analyzing it and propose improvements.
#######################################################################################################

def clean_artist(df, column='artist'):
    # 1. Handle Suffixes/Prefixes
    # Removes "Fotografía de " or " (fotografía)" regardless of case
    df[column] = df[column].str.replace(r'(?i)fotografía de\s*|\s*\(fotografía\)', '', regex=True)

    # 2. Split Multi-Artist rows
    # Splits on ',' or ' y ' and explodes the list into separate rows
    # df[column] = df[column].str.split(r',| y ')
    # df = df.explode(column)
    
    # 3. Standardize Strings
    df[column] = df[column].str.strip()
    
    # 4. Canonical Mapping (The "Sánchez-Toda" Problem)
    # You can expand this dictionary as you find more variants
    mapping = {
        'José Luis Sánchez Toda': 'José Luis Sánchez-Toda',
        'Alfonso López Sánchez Toda': 'Alfonso López Sánchez-Toda',
        'Alonso López Sánchez-Toda': 'Alfonso López Sánchez-Toda' # Probable typo check
    }
    df[column] = df[column].replace(mapping)
    
    return df

unique_artists = df["artist"].dropna().unique()
pd.DataFrame(unique_artists, columns=["artist"]).to_csv("00_unique_artists.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_artists)} unique artists to 00_unique_artists.csv")

df_step1 = clean_artist(df)


✅ Exported 120 unique artists to 00_unique_artists.csv


In [7]:
# Export all unique printers to a csv file. This will be used for analyzing it and propose improvements
#######################################################################################################
import re

def clean_printer(df, column='printer'):
    """
    Standardizes Spanish stamp printer names by removing locations, 
    unifying business titles, and handling overprint (sobrecarga) notes.
    """
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()

    # 1. Standardize FNMT (Handles "Sello", "Timbre", "FNMT-B", etc.)
    fnmt_pattern = r'(?i)Fábrica Nacional del (Sello|Timbre)|Fábrica Nacional de Moneda y Timbre|FNMT(-B)?'
    df[column] = df[column].str.replace(fnmt_pattern, 'Fábrica Nacional de Moneda y Timbre', regex=True)

    # 2. Unify International Printers (Bradbury & Waterlow)
    df[column] = df[column].str.replace(r'(?i)Bradbury Wilkinson (and|&)\s*Co\.?(\s*Ltd\.?)?', 'Bradbury Wilkinson & Co', regex=True)
    df[column] = df[column].str.replace(r'(?i)(Waterlow and Sons|W\.\s*&\s*S\.)(\s*Ltd\.?)?', 'Waterlow and Sons', regex=True)

    # 3. Remove "Talleres", "I.G.", "Hijos de", and generic business terms
    noise_terms = r'(?i)\b(Talleres? de|Tallers? de|I\.G\.|Hijos? de|Hija de|Ltd\.?|en Zonen|Habilitación por)\b'
    df[column] = df[column].str.replace(noise_terms, '', regex=True)

    # 4. Strip Geographical Locations
    # Detects cities preceded by "de", a comma, or inside parentheses
    locations = r'(?i)(,\s*|de\s+|\s*\()(Londres|Madrid|Barcelona|Zúrich|Zurich|Vigo|Granada|Zaragoza|Burgos|Vitoria|Haarlem|Holanda|Tolosa|Castellón|Suiza)(\s*\))?'
    df[column] = df[column].str.replace(locations, '', regex=True)

    # 5. Extract Primary Printer from Overprint notes
    # If "sobre" or "sob." is present, we keep the first entity mentioned
    df[column] = df[column].apply(lambda x: re.split(r'(?i)\s+(sob\.?|sobre|sobrecarga)', str(x))[0])

    # 6. Final Clean-up (double spaces and stray punctuation)
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True).str.strip(' ,.()')
    
    # 7. Specific mappings for consistency
    manual_map = {
        'Orell Fussli': 'Orell Füssli',
        'Oliva Vilanova': 'Oliva de Vilanova'
    }
    df[column] = df[column].replace(manual_map)

    return df


unique_printers = df["printer"].dropna().unique()
pd.DataFrame(unique_printers, columns=["printer"]).to_csv("00_unique_printers.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_printers)} unique printers to 00_unique_printers.csv")

df_step2 = clean_printer(df_step1)


✅ Exported 42 unique printers to 00_unique_printers.csv


In [8]:
# Export all unique print types to a csv file. This will be used for analyzing it and propose 
# improvements
#############################################################################################
import re

def clean_print_type(df, column='print_type'):
    """
    Standardizes printing methods by extracting the primary process 
    and removing overprint/surcharge noise.
    """
    df = df.copy()

    # 1. Normalize abbreviations and "Habilitaciones"
    # Converts "hab." to "Habilitación", "sob." to "Sobre", etc.
    norm_map = {
        r'(?i)\bhab\.?\b': 'Habilitación',
        r'(?i)\bsob\.?\b': 'Sobre',
    }
    for pattern, replacement in norm_map.items():
        df[column] = df[column].str.replace(pattern, replacement, regex=True)

    # 2. Extract the Primary Method
    # We split by delimiters like commas, parentheses, 'y', or 'sobre'
    # and take the first part.
    # Example: "Calcografía, sobreimpresión Tipografía" -> "Calcografía"
    delimiters = r',| \(| y | sobre| habilitación'
    df[column] = df[column].apply(
        lambda x: re.split(delimiters, str(x), flags=re.IGNORECASE)[0].strip()
    )

    # 3. Specific fix for "Habilitación por [Method]" 
    # If the string starts with "Habilitación por", the actual method follows it.
    df[column] = df[column].str.replace(r'(?i)Habilitación por\s+', '', regex=True)

    # 4. Final Polish
    # Title Case (e.g., "offset" -> "Offset") and stripping whitespace
    df[column] = df[column].str.capitalize().str.strip()

    return df

unique_print_types = df["print_type"].dropna().unique()
pd.DataFrame(unique_print_types, columns=["print_type"]).to_csv("00_unique_print_types.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_print_types)} unique print types to 00_unique_print_types.csv")

df_step3 = clean_print_type(df_step2)

✅ Exported 106 unique print types to 00_unique_print_types.csv


In [9]:
# Export all unique perforations to a csv file
# First clean things like 12 3/4 to 12.75
##############################################
import re

def convert_fractions(text):
    if pd.isna(text):
        return text
    
    # Regex to find mixed numbers (e.g., 11 1/2) or simple fractions (1/2)
    # It looks for: (optional integer) + (space) + (numerator)/(denominator)
    def fraction_replacer(match):
        parts = match.group(0).split()
        if len(parts) == 2:  # Mixed number: '11', '1/2'
            num, frac = parts
            div = frac.split('/')
            return str(float(num) + int(div[0]) / int(div[1]))
        else:  # Just a fraction: '1/2'
            div = parts[0].split('/')
            return str(int(div[0]) / int(div[1]))

    # Pattern: \d+ \d+/\d+ (Mixed) or \d+/\d+ (Fraction)
    pattern = r'(\d+\s+\d+/\d+|\d+/\d+)'
    
    # Replace all found fractions with their decimal equivalent
    result = re.sub(pattern, fraction_replacer, str(text))
    
    # Optional: Clean up trailing zeros (e.g., 11.5 instead of 11.5)
    return result

def clean_perforation(df, column='perforation'):
    df = df.copy()

    # 1. Unify Imperforate stamps
    df[column] = df[column].str.replace(r'(?i)Sin dentar|No dentado', 'Imperforado', regex=True)

    # 2. Standardize decimal separators (comma to dot)
    df[column] = df[column].str.replace(',', '.', regex=False)

    # 3. Handle complex descriptions (e.g., "12.5 horizontal por 12.75 vertical")
    # We convert this to "12.5 x 12.75"
    df[column] = df[column].str.replace(r'(?i)(\d+\.?\d*)\s+horizontal\s+por\s+(\d+\.?\d*)\s+vertical', r'\1 x \2', regex=True)
    df[column] = df[column].str.replace(r'(?i)\s+por\s+', ' x ', regex=True)

    # 4. Remove machine types but keep the numbers
    # Removes "de peine", "de línea", "de molde", etc.
    machine_types = r'(?i)\s+(de\s+)?(peine|línea|linea|molde|de\s+molde)'
    df[column] = df[column].str.replace(machine_types, '', regex=True)

    # 5. Handle "or" cases (11 o 11.5)
    # Usually, we take the first one for grouping, or keep it as "11/11.5"
    df[column] = df[column].str.replace(r'\s+(o|ó|y)\s+', ' / ', regex=True)

    # 6. Final Polish
    # Remove parentheses like "(peine)" and extra spaces
    df[column] = df[column].str.replace(r'\(.*?\)', '', regex=True)
    df[column] = df[column].str.strip()
    
    # Standardize 'x' spacing
    df[column] = df[column].str.replace(r'\s*x\s*', ' x ', regex=True)

    return df

# Clean the perforation column
df['perforation'] = df['perforation'].apply(convert_fractions)

unique_perforations = df['perforation'].dropna().unique()
pd.DataFrame(unique_perforations, columns=['perforation']).to_csv('00_unique_perforations.csv', sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_perforations)} unique perforations to 00_unique_perforations.csv")

df_step4 = clean_perforation(df_step3)
last_df = df_step4

✅ Exported 135 unique perforations to 00_unique_perforations.csv


In [10]:
# Export all unique paper types to a csv file. This will be used for analyzing it and propose 
# improvements
#############################################################################################
unique_paper_types = df["paper_type"].dropna().unique()
pd.DataFrame(unique_paper_types, columns=["paper_type"]).to_csv("00_unique_paper_types.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_paper_types)} unique paper types to 00_unique_paper_types.csv")

✅ Exported 27 unique paper types to 00_unique_paper_types.csv


In [11]:
# Export all unique stamp types to a csv file. This will be used for analyzing it and propose 
# improvements
#############################################################################################
unique_stamp_types = df["stamp_type"].dropna().unique()
pd.DataFrame(unique_stamp_types, columns=["stamp_type"]).to_csv("00_unique_stamp_types.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_stamp_types)} unique stamp types to 00_unique_stamp_types.csv")

✅ Exported 49 unique stamp types to 00_unique_stamp_types.csv


In [12]:
# Export all unique colors to a csv file. This will be used for analyzing it and propose 
# improvements
#############################################################################################
unique_color = df["color"].dropna().unique()
pd.DataFrame(unique_color, columns=["color"]).to_csv("00_unique_colors.csv", sep='|', header=False, index=False)
print(f"✅ Exported {len(unique_color)} unique colors to 00_unique_colors.csv")

✅ Exported 835 unique colors to 00_unique_colors.csv


In [13]:
# Remove duplicated lines
#########################
last_df = last_df.drop_duplicates()

In [14]:
# Replace artists with the cleaned artists
##########################################
artists_mapping = pd.read_csv("00_unique_artists_mapping.csv", sep='|')
artists_map = dict(zip(artists_mapping['original'], artists_mapping['mapped_artist']))
mapping = last_df.copy()
mapping['artist'] = mapping['artist'].replace(artists_map)

In [15]:
# Replace colors with the cleaned colors
##########################################
colors_mapping = pd.read_csv("00_unique_colors_mapping.csv", sep='|')
colors_map = dict(zip(colors_mapping['original'], colors_mapping['mapped_color']))
mapping['color'] = mapping['color'].replace(colors_map)

In [16]:
# Replace paper types with the cleaned paper types
##################################################
paper_types_mapping = pd.read_csv("00_unique_paper_types_mapping.csv", sep='|')
paper_types_map = dict(zip(paper_types_mapping['original'], paper_types_mapping['mapped_paper_type']))
mapping['paper_type'] = mapping['paper_type'].replace(paper_types_map)

In [17]:
# Replace perforations with the cleaned perforations
####################################################
perforations_mapping = pd.read_csv("00_unique_perforations_mapping.csv", sep='|')
perforations_map = dict(zip(perforations_mapping['original'], perforations_mapping['mapped_perforation']))
mapping['perforation'] = mapping['perforation'].replace(perforations_map)

In [18]:
# Replace print types with the cleaned print types
##################################################
print_types_mapping = pd.read_csv("00_unique_print_types_mapping.csv", sep='|')
print_types_map = dict(zip(print_types_mapping['original'], print_types_mapping['mapped_print_type']))
mapping['print_type'] = mapping['print_type'].replace(print_types_map) 

In [19]:
# Replace printers with the cleaned printers
############################################
printers_mapping = pd.read_csv("00_unique_printers_mapping.csv", sep='|')
printers_map = dict(zip(printers_mapping['original'], printers_mapping['mapped_printer']))
mapping['printer'] = mapping['printer'].replace(printers_map)

In [20]:
# Replace stamp types with the cleaned stamp types
##################################################
stamp_types_mapping = pd.read_csv("00_unique_stamp_types_mapping.csv", sep='|')
stamp_types_map = dict(zip(stamp_types_mapping['original'], stamp_types_mapping['mapped_stamp_type']))
mapping['stamp_type'] = mapping['stamp_type'].replace(stamp_types_map)

In [21]:
# Save cleaned csv to a new one
###############################
last_df = mapping.copy()
last_df.to_csv(CLEAN_CSV_FILE, sep="|", header=True, index=False)